In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

from delta.tables import DeltaTable


# ============================================================
# Configuration
# ============================================================

BRONZE_TABLE = "dev_catalog.bronze.customer"

SILVER_TABLE = "dev_catalog.silver.customer"

REJECT_TABLE = "dev_catalog.silver.customer_reject"

LOG_TABLE = "dev_catalog.control.pipeline_run_log"

PIPELINE_NAME = "customer_etl_job"
TASK_NAME = "silver_customer"


# ============================================================
# Generate Silver task run ID
# ============================================================

silver_run_id = str(
    spark.sql("SELECT uuid()").first()[0]
)

start_timestamp = spark.sql(
    "SELECT current_timestamp()"
).first()[0]


print(
    f"Starting Silver processing. "
    f"Silver Run ID: {silver_run_id}"
)


# ============================================================
# Pipeline Log Schema
# ============================================================

log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("records_rejected", LongType(), True),
    StructField("error_message", StringType(), True)
])


# ============================================================
# Silver Processing
# ============================================================

try:

    print("Starting incremental Silver customer processing...")

    #raise Exception("TEST_RETRY_FAILURE")
    # ========================================================
    # Identify the latest successful Bronze run
    # ========================================================

    latest_bronze_run = spark.sql(
        f"""
        SELECT run_id
        FROM {LOG_TABLE}
        WHERE pipeline_name = '{PIPELINE_NAME}'
          AND task_name = 'bronze_customer'
          AND status = 'SUCCESS'
        ORDER BY end_timestamp DESC
        LIMIT 1
        """
    )


    bronze_run_rows = latest_bronze_run.collect()


    if len(bronze_run_rows) == 0:

        print(
            "No successful Bronze run found. "
            "Nothing to process."
        )

        records_processed = 0
        records_rejected = 0

    else:

        current_bronze_run_id = bronze_run_rows[0]["run_id"]

        print(
            f"Processing Bronze Run ID: "
            f"{current_bronze_run_id}"
        )


        # ====================================================
        # Read ONLY records from the latest Bronze run
        # ====================================================

        bronze_df = (
            spark.table(BRONZE_TABLE)
            .filter(
                F.col("_run_id") == current_bronze_run_id
            )
        )


        # ====================================================
        # Identify rejected records
        # ====================================================

 
        reject_df = (
            bronze_df
            .filter(
                F.col("Customer_ID").isNull()
            )
            .drop("_run_id")
            .withColumn(
                "_reject_reason",
                F.lit("Customer_ID is null")
            )
            .withColumn(
                "_reject_timestamp",
                F.current_timestamp()
             )
        )


        # ====================================================
        # Count rejected records
        # ====================================================

        records_rejected = reject_df.count()


        # ====================================================
        # Write rejected records
        # ====================================================

        if records_rejected > 0:

            (
                reject_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(REJECT_TABLE)
            )

            print(
                f"Rejected records: {records_rejected}"
            )


        # ====================================================
        # Valid records
        # ====================================================

        valid_df = (
            bronze_df
            .filter(
                F.col("Customer_ID").isNotNull()
            )
        )


        # ====================================================
        # Clean records
        # ====================================================

        cleaned_df = (
            valid_df

            .withColumn(
                "Customer_Name",
                F.trim(F.col("Customer_Name"))
            )

            .withColumn(
                "City",
                F.trim(F.col("City"))
            )

            .withColumn(
                "_silver_updated_timestamp",
                F.current_timestamp()
            )
        )


        # ====================================================
        # Keep latest record for each Customer_ID
        # ====================================================

        from pyspark.sql.window import Window

        window_spec = (
            Window
            .partitionBy("Customer_ID")
            .orderBy(
                F.col("Modified_Date").desc_nulls_last(),
                F.col("_ingestion_timestamp").desc()
            )
        )


        silver_updates = (
            cleaned_df
            .withColumn(
                "_row_number",
                F.row_number().over(window_spec)
            )
            .filter(
                F.col("_row_number") == 1
            )
            .drop("_row_number")
        )


        # ====================================================
        # Count records
        # ====================================================

        records_processed = silver_updates.count()


        # ====================================================
        # MERGE INTO SILVER
        # ====================================================

        if records_processed > 0:

            silver_delta = DeltaTable.forName(
                spark,
                SILVER_TABLE
            )


            (
                silver_delta.alias("target")

                .merge(
                    silver_updates.alias("source"),

                    "target.Customer_ID = source.Customer_ID"
                )

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .execute()
            )


            print(
                "Silver MERGE completed successfully."
            )

        else:

            print(
                "No valid records available for Silver MERGE."
            )


    # ========================================================
    # End timestamp
    # ========================================================

    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]


    # ========================================================
    # SUCCESS LOG
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            silver_run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "SUCCESS",
            records_processed,
            records_rejected,
            None
        )],
        schema=log_schema
    )


    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        "Incremental Silver processing completed successfully."
    )


# ============================================================
# Failure handling
# ============================================================

except Exception as e:

    end_timestamp = spark.sql(
        "SELECT current_timestamp()"
    ).first()[0]

    error_message = str(e)


    # ========================================================
    # Write FAILED log
    # ========================================================

    log_df = spark.createDataFrame(
        [(
            silver_run_id,
            PIPELINE_NAME,
            TASK_NAME,
            start_timestamp,
            end_timestamp,
            "FAILED",
            0,
            0,
            error_message
        )],
        schema=log_schema
    )


    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )


    print(
        f"Silver processing failed: {error_message}"
    )


    raise